# SIMURGH GMD — Inference & Calculator

Loads the best checkpoint from the GMD FragmentDuplication training run and
provides a ready-to-use `GOALCalculator` for ASE.

**Checkpoint:** `epoch=0000-val_loss=0.0711.ckpt`  
**Run:** `logs/KRONOS-GMD/runs/2026-06-01_23-40-34_trajectory_kronos`  
**Dataset:** GMD FragmentDuplication — O₂CₙH₂ₙ molecules, PBE/def2-SVP DFT

In [ ]:
from __future__ import annotations

from pathlib import Path

import torch

# Trigger GOAL registry population and Lightning checkpoint patch
import goal.ml.cli  # noqa: F401

from goal.ml.utils.calculator import GOALCalculator

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 1 · Load the Best Checkpoint

The best checkpoint is selected by lowest validation loss.  
`GOALCalculator` reconstructs the backbone + head from the saved Hydra config
and loads the state dict automatically.

In [ ]:
# ── Paths ───────────────────────────────────────────────────────────────────
PROJECT_ROOT = Path("..")
RUN_DIR = PROJECT_ROOT / "logs" / "monolithic_arace_gmd26" / "runs" / "2026-06-17_12-23-42_trajectory_monolithic_arace_fragment_duplication"
CKPT_DIR = RUN_DIR / "checkpoints"

assert CKPT_DIR.is_dir(), f"Checkpoint directory not found: {CKPT_DIR.resolve()}"

# ── Device selection ─────────────────────────────────────────────────────────
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Checkpoint dir  : {CKPT_DIR.resolve()}")
print(f"Running on      : {DEVICE}")

In [ ]:
# ── Build the calculator ──────────────────────────────────────────────────────
# GOALCalculator reads checkpoint_state.json to pick the best top-k checkpoint,
# then loads via the frozen source snapshot — immune to later code changes.
# Passing the directory (FORMAT B) instead of a hardcoded filename means this
# cell always loads the true best checkpoint for this run.

calc = GOALCalculator(
    checkpoint_path=CKPT_DIR,
    checkpoint="best",
    device=DEVICE,
)

print(calc)
n_params = sum(p.numel() for p in calc._module.parameters())
print(f"Model parameters: {n_params:,}")

## 1.5 · Quick test

In [ ]:
import ase.io
TRAJ_PATH  = "/home/masoud/Projects/SCAI/GMD/data/CarboxylicAcids/O2C7H14_PBE.traj"
# Read all frames and pick a middle snapshot
traj  = ase.io.read(TRAJ_PATH, index=":")
frame = traj[1800]

# Reference (DFT) forces are stored in the SinglePointCalculator attached to each frame
ref_forces = frame.get_forces().copy()

print(f"Trajectory length : {len(traj)} frames")
print(f"Using frame index : {len(traj) // 2}")
print(f"Atoms             : {frame.get_chemical_formula()}  ({len(frame)} atoms)")
print(f"Ref forces shape  : {ref_forces.shape}")

atoms_pred = frame.copy()
atoms_pred.calc = calc
pred_forces = atoms_pred.get_forces()

print(f"Predicted forces shape: {pred_forces.shape}")

In [ ]:
import numpy as np
# Per-atom error: L2 norm of (pred - ref) force vector
atom_errors = np.linalg.norm(pred_forces - ref_forces, axis=1)
component_mae = np.abs(pred_forces - ref_forces).mean()  
# newton_error = pred_forces.norm(dim=-1).mean().item() - ref_forces.norm(dim=-1).mean().item()

print(f"Force errors (eV/Å):")
print(f"  min  = {atom_errors.min():.4f}")
print(f"  max  = {atom_errors.max():.4f}")
print(f"  mean = {atom_errors.mean():.4f}")
print(f"  component MAE = {component_mae:.4f}")
# print(f"  newton = {newton_error:.4f}")

# Normalize to [0, 1]  (0 = best / white, 1 = worst / red)
err_range   = atom_errors.max() - atom_errors.min()
norm_errors = (atom_errors - atom_errors.min()) / (err_range + 1e-10)

print(f"Total force error (eV/Å): {atom_errors.sum():.4f}")

In [ ]:
import torch
from goal.ml.training.metrics import mlip_metrics

# Convert numpy arrays to tensors; single structure → all atoms belong to graph 0
pred_f_t  = torch.as_tensor(pred_forces, dtype=torch.float32)
ref_f_t   = torch.as_tensor(ref_forces,  dtype=torch.float32)
batch_idx = torch.zeros(len(pred_forces), dtype=torch.long)

# mlip_metrics expects a predictions dict and a batch-like object with
# .forces and .batch — wrap the single-structure arrays minimally.
class _SingleStructureBatch:
    forces = ref_f_t
    batch  = batch_idx

metrics = mlip_metrics({"forces": pred_f_t}, _SingleStructureBatch())
newton_violation = metrics["newton_violation"].item()

net_force = pred_forces.sum(axis=0)  # same as scatter_sum over a single graph
magnitude_bias = pred_f_t.norm(dim=-1).mean().item() - ref_f_t.norm(dim=-1).mean().item()

print(f"Newton violation (project formula) : {newton_violation:.4e} eV/Å")
print(f"  net force vector : [{net_force[0]:+.4e}  {net_force[1]:+.4e}  {net_force[2]:+.4e}] eV/Å")
print()
print(f"Magnitude bias (old commented formula) : {magnitude_bias:+.4e} eV/Å")
print(f"  (mean|F_pred| − mean|F_ref| — NOT a Newton metric)")

## 2 · Quick Sanity Check

Attach the calculator to an ASE `Atoms` object and verify energy + forces.

In [ ]:
import numpy as np
from ase import Atoms
from ase import io

# Water molecule — H, O, H all within the training element set {H, C, O}
atoms: Atoms = io.read("../data/GMD/FragmentDuplication/TestOOD/O4C4H6_PBE.traj", index=0)
atoms.calc = calc

energy: float = atoms.get_potential_energy()   # eV
forces: np.ndarray = atoms.get_forces()         # eV/Å,  shape (N, 3)

print(f"Formula  : {atoms.get_chemical_formula()}")
print(f"Energy   : {energy:.4f} eV")
print()
print("Forces (eV/Å):")
for sym, f in zip(atoms.get_chemical_symbols(), forces):
    print(f"  {sym:2s}  [{f[0]:+8.4f}  {f[1]:+8.4f}  {f[2]:+8.4f}]")

## 3 · Evaluate on a Test-Set Structure

Load one trajectory from the OOD test split and compare predicted vs. DFT values.

In [ ]:
from ase.io import read as ase_read

TEST_DIR = PROJECT_ROOT / "data" / "GMD" / "FragmentDuplication" / "TestOOD"
traj_files = sorted(TEST_DIR.glob("*.traj"))
print(f"Found {len(traj_files)} test trajectory files in {TEST_DIR.name}/")

# Pick the first structure of the first file
sample_frame: Atoms = ase_read(str(traj_files[0]), index=0)
print(f"\nSample structure: {sample_frame.get_chemical_formula()} ({len(sample_frame)} atoms)")

# DFT reference values
dft_energy: float = sample_frame.get_potential_energy()   # already stored in .traj
dft_forces: np.ndarray = sample_frame.get_forces()

# SIMURGH prediction
sample_frame.calc = calc
pred_energy: float = sample_frame.get_potential_energy()
pred_forces: np.ndarray = sample_frame.get_forces()

n_atoms = len(sample_frame)
energy_mae_per_atom = abs(pred_energy - dft_energy) / n_atoms
forces_mae = np.abs(pred_forces - dft_forces).mean()

print(f"\n{'':20s} {'DFT':>12s} {'SIMURGH':>12s}")
print(f"{'Energy (eV)':20s} {dft_energy:12.4f} {pred_energy:12.4f}")
print(f"{'Energy MAE/atom':20s} {'':12s} {energy_mae_per_atom:12.4f} eV/atom")
print(f"{'Forces MAE (eV/Å)':20s} {'':12s} {forces_mae:12.4f}")

In [ ]:
from ase.io import read as ase_read

TEST_DIR = PROJECT_ROOT / "data" / "GMD" / "FragmentDuplication" / "TestID"
traj_files = sorted(TEST_DIR.glob("*.traj"))
print(f"Found {len(traj_files)} test trajectory files in {TEST_DIR.name}/")

# Pick the first structure of the first file
sample_frame: Atoms = ase_read(str(traj_files[0]), index=65)
print(f"\nSample structure: {sample_frame.get_chemical_formula()} ({len(sample_frame)} atoms)")

# DFT reference values
dft_energy: float = sample_frame.get_potential_energy()   # already stored in .traj
dft_forces: np.ndarray = sample_frame.get_forces()

# SIMURGH prediction
sample_frame.calc = calc
pred_energy: float = sample_frame.get_potential_energy()
pred_forces: np.ndarray = sample_frame.get_forces()

n_atoms = len(sample_frame)
energy_mae_per_atom = abs(pred_energy - dft_energy) / n_atoms
forces_mae = np.abs(pred_forces - dft_forces).mean()

print(f"\n{'':20s} {'DFT':>12s} {'SIMURGH':>12s}")
print(f"{'Energy (eV)':20s} {dft_energy:12.4f} {pred_energy:12.4f}")
print(f"{'Energy MAE/atom':20s} {'':12s} {energy_mae_per_atom:12.4f} eV/atom")
print(f"{'Forces MAE (eV/Å)':20s} {'':12s} {forces_mae:12.4f}")

In [ ]:
# Load MACE and predict on a copy so the original SinglePointCalculator is preserved
sample_frame: Atoms = ase_read(str(traj_files[0]), index=65)
ref_forces = sample_frame.get_forces().copy()
atoms_pred = sample_frame.copy()
atoms_pred.calc = calc
pred_forces = atoms_pred.get_forces()

print(f"Predicted forces shape: {pred_forces.shape}")

In [ ]:
# Per-atom error: L2 norm of (pred - ref) force vector
atom_errors = np.linalg.norm(pred_forces - ref_forces, axis=1)

print(f"Force errors (eV/Å):")
print(f"  min  = {atom_errors.min():.4f}")
print(f"  max  = {atom_errors.max():.4f}")
print(f"  mean = {atom_errors.mean():.4f}")

# Normalize to [0, 1]  (0 = best / white, 1 = worst / red)
err_range   = atom_errors.max() - atom_errors.min()
norm_errors = (atom_errors - atom_errors.min()) / (err_range + 1e-10)

print(f"Total force error (eV/Å): {atom_errors.sum():.4f}")

sum_pred_forces = pred_forces.sum(axis=0)
sum_ref_forces  = ref_forces.sum(axis=0)
sum_diff        = sum_pred_forces - sum_ref_forces

print("Summed forces (vector):")
print(f"  predicted : {sum_pred_forces}")
print(f"  reference : {sum_ref_forces}")
print(f"  difference: {sum_diff}")

print("\nSummed force magnitudes:")
print(f"  |sum_pred| = {np.linalg.norm(sum_pred_forces):.16f}")
print(f"  |sum_ref|  = {np.linalg.norm(sum_ref_forces):.16f}")
print(f"  |sum_diff| = {np.linalg.norm(sum_diff):.16f}")

In [ ]:
import torch.nn.functional as F_func
import py3Dmol

def error_to_hex(t: float) -> str:
    """Map normalised error t∈[0,1] → hex colour: white (t=0) to red (t=1)."""
    r = 255
    g = int(255 * (1.0 - t))
    b = int(255 * (1.0 - t))
    return f"#{r:02x}{g:02x}{b:02x}"

atom_colors = [error_to_hex(t) for t in norm_errors]

# Quick sanity check — print first few atoms with their error and color
for i, (sym, err, col) in enumerate(zip(sample_frame.get_chemical_symbols(), atom_errors, atom_colors)):
    print(f"  atom {i:2d}  {sym}  error={err:.4f}  color={col}")

# ── Two-sphere helpers ─────────────────────────────────────────────────────────
# inner sphere : relative magnitude error  ||F_pred|-|F_ref|| / ||F_ref||   (white → pink)
# outer shell  : cosine similarity         cos(F_pred, F_ref)
#                cos=-1 → YELLOW  (opacity ~0.85)
#                cos= 0 → RED     (opacity ~0.55)
#                cos=+1 → WHITE   (opacity ~0.25 — faint translucent halo)
#                Always rendered so the shell is visible at every atom.
import matplotlib.colors as mcolors
from matplotlib.colors import LinearSegmentedColormap

# white → pink — relative magnitude error
cmap_mag = LinearSegmentedColormap.from_list("mag_err", ["#ffffff", "#ffb6d5", "#ff1493"])

# Parameterise by t = (1 - cos) / 2  ∈ [0, 1]   (t=0 at cos=+1, t=1 at cos=-1)
#   t=0   (cos=+1): white
#   t=0.5 (cos= 0): red
#   t=1   (cos=-1): yellow
OUTER_WHITE  = "#ffffff"
OUTER_RED    = "#e60026"
OUTER_YELLOW = "#ffd000"
cmap_dir = LinearSegmentedColormap.from_list(
    "dir_err",
    [(0.0, OUTER_WHITE), (0.5, OUTER_RED), (1.0, OUTER_YELLOW)],
)
OUTER_OPACITY_MIN = 0.25
OUTER_OPACITY_MAX = 0.85

def two_sphere_colors(pred_forces, ref_forces, mag_vmax=None):
    """Return (inner_colors, outer_colors, outer_opacities, rel_mag, cos_sim)."""
    vp = torch.as_tensor(pred_forces, dtype=torch.float32)
    vr = torch.as_tensor(ref_forces,  dtype=torch.float32)
    cos_sim = F_func.cosine_similarity(vp, vr, dim=1).cpu().numpy()
    cos_sim = np.nan_to_num(cos_sim, nan=0.0)

    n_pred = np.linalg.norm(pred_forces, axis=1)
    n_ref  = np.linalg.norm(ref_forces,  axis=1)
    rel_mag = np.abs(n_pred - n_ref) / (n_ref + 1e-10)

    vmax = mag_vmax if mag_vmax is not None else max(float(rel_mag.max()), 1e-10)
    inner_colors    = [mcolors.to_hex(cmap_mag(min(float(m) / vmax, 1.0))) for m in rel_mag]

    t_dir           = [(1.0 - float(c)) / 2.0 for c in cos_sim]                 # 0..1
    outer_colors    = [mcolors.to_hex(cmap_dir(t)) for t in t_dir]
    span            = OUTER_OPACITY_MAX - OUTER_OPACITY_MIN
    outer_opacities = [OUTER_OPACITY_MIN + t * span for t in t_dir]
    return inner_colors, outer_colors, outer_opacities, rel_mag, cos_sim

def add_two_spheres(view, positions, inner_colors, outer_colors, outer_opacities,
                    r_inner=0.22, r_outer=0.42):
    """Two concentric spheres per atom; outer shell ALWAYS rendered."""
    for i, pos in enumerate(positions):
        x, y, z = float(pos[0]), float(pos[1]), float(pos[2])
        view.addSphere({
            "center":  {"x": x, "y": y, "z": z},
            "radius":  r_outer,
            "color":   outer_colors[i],
            "opacity": float(outer_opacities[i]),
        })
        view.addSphere({
            "center": {"x": x, "y": y, "z": z},
            "radius": r_inner,
            "color":  inner_colors[i],
        })


In [ ]:
# Serialise the frame to XYZ string for py3Dmol
import io as sio
buf = sio.StringIO()
io.write(buf, sample_frame, format='xyz')
xyz_str = buf.getvalue()

In [ ]:
from ase.neighborlist import natural_cutoffs, NeighborList

symbols   = sample_frame.get_chemical_symbols()
positions = sample_frame.get_positions()

view = py3Dmol.view(width=800, height=550)
view.addModel(xyz_str, 'xyz')

# Suppress py3Dmol's own bond drawing — we draw bonds manually below
view.setStyle({}, {})

# Compute bonds with ASE natural cutoffs (covalent radii, 10% tolerance)
cutoffs = natural_cutoffs(sample_frame, mult=1.1)
nl = NeighborList(cutoffs, self_interaction=False, bothways=False)
nl.update(sample_frame)

for i in range(len(sample_frame)):
    indices, _ = nl.get_neighbors(i)
    for j in indices:
        if j > i:
            p1, p2 = positions[i], positions[j]
            view.addCylinder({
                'start':   {'x': float(p1[0]), 'y': float(p1[1]), 'z': float(p1[2])},
                'end':     {'x': float(p2[0]), 'y': float(p2[1]), 'z': float(p2[2])},
                'radius':  0.08,
                'color':   '#888888',
                'fromCap': True,
                'toCap':   True,
            })

# Two-sphere atoms: inner = magnitude error (white→pink), outer shell = cos sim (yellow/red, opacity ∝ |cos|)
inner_full, outer_full, opac_full, rel_mag_full, cos_full = two_sphere_colors(pred_forces, ref_forces)
add_two_spheres(view, positions, inner_full, outer_full, opac_full)

for sym, pos in zip(symbols, positions):
    view.addLabel(sym, {
        'position':       {'x': float(pos[0]), 'y': float(pos[1]), 'z': float(pos[2])},
        'fontSize':       12,
        'fontColor':      'black',
        'showBackground': False,
        'alignment':      'center',
    })

view.zoomTo()
view.show()


In [ ]:
# Two colourbars: magnitude error (inner sphere) and cosine similarity (outer shell)
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.cm import ScalarMappable

# Inner: relative magnitude error
fig, ax = plt.subplots(figsize=(5, 0.6))
fig.subplots_adjust(bottom=0.5)
norm_mag = mcolors.Normalize(vmin=0.0, vmax=float(rel_mag_full.max()))
cb_mag   = fig.colorbar(ScalarMappable(norm=norm_mag, cmap=cmap_mag), cax=ax,
                        orientation='horizontal')
cb_mag.set_label('Inner sphere — relative magnitude error '
                 r'$|\,\|F_{pred}\|-\|F_{ref}\|\,| / \|F_{ref}\|$', fontsize=10)
plt.show()

# Outer: cosine similarity ∈ [-1, +1]
#   cos=-1 → yellow (opaque)
#   cos= 0 → red    (semi-translucent)
#   cos=+1 → white  (faint translucent halo)
cmap_dir_cb = mcolors.LinearSegmentedColormap.from_list(
    'dir_err_cb',
    [(0.0, OUTER_YELLOW), (0.5, OUTER_RED), (1.0, OUTER_WHITE)],   # cos=-1 → 0 → +1
)
fig, ax = plt.subplots(figsize=(5, 0.6))
fig.subplots_adjust(bottom=0.5)
norm_dir = mcolors.Normalize(vmin=-1.0, vmax=1.0)
cb_dir   = fig.colorbar(ScalarMappable(norm=norm_dir, cmap=cmap_dir_cb), cax=ax,
                        orientation='horizontal')
cb_dir.set_label(r'Outer shell — cosine similarity  $\cos(F_{pred}, F_{ref})$'
                 r'   (opacity grows from 0.25 at $\cos=+1$ to 0.85 at $\cos=-1$)',
                 fontsize=10)
plt.show()


In [ ]:
from pathlib import Path

split_dirs = {
    "OOD": PROJECT_ROOT / "data" / "GMD" / "FragmentDuplication" / "TestOOD",
    "ID":  PROJECT_ROOT / "data" / "GMD" / "FragmentDuplication" / "TestID",
}

def evaluate_split(split_dir):
    traj_paths = sorted(split_dir.glob("*.traj"))
    split_results = []
    for traj_path in traj_paths:
        frames = ase_read(str(traj_path), index=":")
        for idx, frame in enumerate(frames):
            dft_energy = frame.get_potential_energy()
            dft_forces = frame.get_forces()
            n_atoms = len(frame)

            frame.calc = calc
            pred_energy = frame.get_potential_energy()
            pred_forces = frame.get_forces()

            energy_mae_per_atom = abs(pred_energy - dft_energy)
            forces_mae = np.abs(pred_forces - dft_forces).mean()

            mag_pred = np.linalg.norm(pred_forces, axis=1)
            mag_ref = np.linalg.norm(dft_forces, axis=1)
            cos_sim = np.einsum("ij,ij->i", pred_forces, dft_forces) / (
                mag_pred * mag_ref + 1e-12
            )

            split_results.append({
                "file": traj_path.name,
                "index": idx,
                "n_atoms": n_atoms,
                "energy_mae_per_atom": energy_mae_per_atom,
                "forces_mae": forces_mae,
                "forces_mag_pred_mean": float(mag_pred.mean()),
                "forces_mag_ref_mean": float(mag_ref.mean()),
                "forces_cos_mean": float(cos_sim.mean()),
            })
    return split_results

all_results = {}
for split_name, split_dir in split_dirs.items():
    results = evaluate_split(split_dir)
    all_results[split_name] = results

for split_name, results in all_results.items():
    energy_mae = np.mean([r["energy_mae_per_atom"] for r in results])
    forces_mae = np.mean([r["forces_mae"] for r in results])
    mag_pred = np.mean([r["forces_mag_pred_mean"] for r in results])
    mag_ref = np.mean([r["forces_mag_ref_mean"] for r in results])
    cos_mean = np.mean([r["forces_cos_mean"] for r in results])

    print(f"=== {split_name} test ===")
    print(f"Structures evaluated : {len(results)}")
    print(f"Energy MAE / atom    : {energy_mae:.6f} eV/atom")
    print(f"Forces MAE           : {forces_mae:.6f} eV/Å")
    print(f"Pred. force mag mean : {mag_pred:.6f} eV/Å")
    print(f"Ref. force mag mean  : {mag_ref:.6f} eV/Å")
    print(f"Force cosine mean    : {cos_mean:.6f}")
    print()

    for r in results:
        print(
            f"{r['file']}[{r['index']}]  "
            f"E_mae/atom={r['energy_mae_per_atom']:.6f}  "
            f"F_mae={r['forces_mae']:.6f}  "
            f"|F_pred|={r['forces_mag_pred_mean']:.6f}  "
            f"|F_ref|={r['forces_mag_ref_mean']:.6f}  "
            f"cos={r['forces_cos_mean']:.6f}"
        )
    print()

## 4 · Geometry Optimisation

Use ASE's BFGS optimiser to relax a perturbed structure to the nearest energy minimum.

In [ ]:
from ase.optimize import BFGS

# Fresh copy — perturb positions slightly to start away from equilibrium
opt_atoms: Atoms = sample_frame.copy()
rng = np.random.default_rng(seed=42)
opt_atoms.positions += rng.normal(scale=0.05, size=opt_atoms.positions.shape)
opt_atoms.calc = calc

e0 = opt_atoms.get_potential_energy()
f0_max = np.abs(opt_atoms.get_forces()).max()
print(f"Initial energy   : {e0:.4f} eV")
print(f"Initial max force: {f0_max:.4f} eV/Å")

opt = BFGS(opt_atoms, logfile=None)
converged = opt.run(fmax=0.05, steps=100)

ef = opt_atoms.get_potential_energy()
ff_max = np.abs(opt_atoms.get_forces()).max()
print(f"\nFinal energy     : {ef:.4f} eV  (ΔE = {ef - e0:.4f} eV)")
print(f"Final max force  : {ff_max:.4f} eV/Å")
print(f"Steps taken      : {opt.nsteps}")
print(f"Converged        : {converged}")

## 5 · Short MD Simulation

NVT Langevin dynamics at 300 K using the SIMURGH potential.

In [ ]:
from ase import units
from ase.md.langevin import Langevin
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution

md_atoms: Atoms = sample_frame.copy()
md_atoms.calc = calc

MaxwellBoltzmannDistribution(md_atoms, temperature_K=300)

dyn = Langevin(
    md_atoms,
    timestep=0.5 * units.fs,
    temperature_K=300,
    friction=0.01 / units.fs,
)

md_energies: list[float] = []
dyn.attach(lambda: md_energies.append(md_atoms.get_potential_energy()), interval=1)

print("Running 50-step Langevin MD at 300 K ...")
dyn.run(steps=50)
print(f"Done — {len(md_energies)} energy snapshots collected")
print(f"Energy range: {min(md_energies):.4f} → {max(md_energies):.4f} eV")

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(md_energies, linewidth=1.5, color="#2196F3")
ax.set_xlabel("MD step")
ax.set_ylabel("Potential energy (eV)")
ax.set_title(f"SIMURGH MD — {md_atoms.get_chemical_formula()} @ 300 K")
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

## 6 · Batch Inference on the Test Set

Run the model directly on PyG batches for fast bulk evaluation (no ASE overhead).

In [ ]:
import torch
from goal.ml.data.datamodule import GOALDataModule
from goal.ml.training.archive import resolve_checkpoint

# Resolve the same best checkpoint the calculator uses
best_ckpt_path = resolve_checkpoint(CKPT_DIR, "best")
print(f"Best checkpoint : {best_ckpt_path.name}")

raw_ckpt = torch.load(str(best_ckpt_path), map_location="cpu", weights_only=False)
cfg = raw_ckpt["hyper_parameters"]["config"]

dm = GOALDataModule(cfg)
dm.prepare_data()
dm.setup("test")
test_loader = dm.test_dataloader()

print(f"Test set size : {len(dm.data_test):,} structures")
print(f"Batch size    : {cfg.data.batch_size}")
print(f"Num batches   : {len(test_loader)}")

In [ ]:
module = calc._module
module.eval()

all_e_mae_per_atom: list[float] = []
all_f_mae: list[float] = []

with torch.enable_grad():
    for batch in test_loader:
        batch = batch.to(DEVICE)
        with torch.no_grad():
            pass  # just move to device
        preds = module(batch)

        e_pred = preds["energy"].detach().cpu()       # (B,)
        e_true = batch.energy.cpu()                    # (B,)
        n_atoms = preds["num_atoms"].detach().cpu()   # (B,)

        e_mae = (torch.abs(e_pred - e_true) / n_atoms).mean().item()
        all_e_mae_per_atom.append(e_mae)

        if "forces" in preds and batch.forces is not None:
            f_pred = preds["forces"].detach().cpu()
            f_true = batch.forces.cpu()
            f_mae = torch.abs(f_pred - f_true).mean().item()
            all_f_mae.append(f_mae)

print("=== Test set results (OOD) ===")
print(f"Energy MAE/atom : {sum(all_e_mae_per_atom) / len(all_e_mae_per_atom):.4f} eV/atom")
if all_f_mae:
    print(f"Forces MAE      : {sum(all_f_mae) / len(all_f_mae):.4f} eV/Å")

---

## Summary

| Item | Value |
|---|---|
| Checkpoint | `epoch=0000-val_loss=0.0711.ckpt` |
| Cutoff | 8.0 Å |
| Elements | H, C, O |
| Backbone | SIMURGH (lmax=2, 2 interactions, 32 channels) |
| Head | `EnergyForcesHead` (autograd forces) |

The `calc` object above is a standard ASE calculator — plug it into any ASE workflow.